# 05. Replay analysis and explaining the project

UTS 43008 · Assignment 3 · Project 12 · Dihedrals

Connect the game on screen with the code using one recorded development episode. The web app,
baselines and PPO step the same Python environment. The canvas displays returned state;
it is not a second learning environment.

This notebook runs a heuristic episode, not a trained PPO strategy.

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. Record an episode

Replays store actions, snapshots, reward components and combat events. They help debug and
explain failures, but one episode is not a statistical comparison.

In [ ]:
from agents.heuristic import HeuristicCommander
from experiments.rollout import play_episode, save_replay, load_replay

board = SiegeEnv(base_seed=10200, motifs=("closed_ring",), record=True)
replay = play_episode(board, HeuristicCommander())
board.close()
save_replay(replay, OUTPUT / "heuristic-replay.json")
assert load_replay(OUTPUT / "heuristic-replay.json") == json.loads(json.dumps(replay))
display(pd.Series(replay["summary"]))

## 2. Beginning, middle and end

Look for deployment direction, damage taken before reaching defences, wall breaches and whether
the Town Hall survives. These observations suggest development questions, not a reason to inspect
protected final layouts.

In [ ]:
snapshots = replay["snapshots"]
indices = [0, len(snapshots)//2, len(snapshots)-1]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for index, ax in zip(indices, axes):
    draw_snapshot(snapshots[index], ax, style="presentation", show_hud=False)
    ax.set_title(f"Frame {index} · tick {snapshots[index]['tick']}")
plt.tight_layout()
save_figure("replay-story")

## 3. Follow actions and rewards

Buying changes reserves and spend. Finishing shopping enters battle and charges army cost.
Deployment adds a troop and advances combat. Waiting advances combat without deployment.
Each milestone pays only on the transition when it is first achieved.

In [ ]:
trace = pd.DataFrame([{"decision": i, "action": describe_action(frame["last_action"]),
    "tick": frame["tick"], "destruction": frame["destruction"], "return": frame["episode_return"],
    **frame["reward_components"]} for i, frame in enumerate(snapshots) if frame["last_action"] is not None])
display(trace.head(12))
display(trace.tail(5))
components = trace[list(C.REWARD_COMPONENT_KEYS)].sum()
display(components.rename("episode reward total"))
assert np.isclose(components.sum(), replay["summary"]["episode_return"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].step(trace.decision, trace.destruction, where="post")
axes[0].set(xlabel="Agent decisions", ylabel="Destroyed value fraction", ylim=(0, 1))
axes[1].step(trace.decision, trace["return"], where="post")
axes[1].set(xlabel="Agent decisions", ylabel="Cumulative return")
plt.tight_layout()
save_figure("replay-progress")

## 4. Trace one click through the architecture

| Layer | Responsibility |
|---|---|
| ui/app.py and canvas | Read actions and draw state |
| env/siege_env.py | Check legality, advance the episode and calculate reward |
| env/combat.py | Move, choose attacks, apply simultaneous damage and check endings |
| env/pathfinding.py | Weighted four-neighbour routes |
| train/features.py and PPO | Convert observation into a learned action |
| experiments/rollout.py | Consistent episode loop for policy evaluation |
| train/checkpoints.py | Save enough state for continuation |

Python is authoritative for the current app. Vaibhav's visual theme is retained. His preserved
prototype is reference material, not another source of assignment results.

## 5. A short, honest pitch

We are studying whether rotation and reflection augmentation helps an RL commander generalise.
It buys troops and chooses deployment positions in a small base-attack game. Movement and combat
follow fixed rules. We compare the same Maskable PPO learner with and without augmentation,
alongside random and scripted baselines.

The simulator, web demonstration, training and evaluation code are implemented. Short software
checks do not establish a strong strategy. Long matched-seed experiments and final findings
still need to be completed.

## 6. Questions the professor may ask

| Question | Plain-English answer |
|---|---|
| What is learned? | Army purchases, deployment positions and timing. Troop movement is scripted. |
| Why RL? | Early purchases and deployment choices affect later combat rewards. |
| Why PPO? | A practical policy-gradient baseline; masking handles phase-dependent legal actions. |
| What is custom? | The game/interface, feature fusion, D4 mapping and experiment/recovery workflow. |
| Why two phases? | Learn deployment with a fixed army, then joint purchasing and deployment. |
| Is the observation Markov? | Not fully. Some per-troop internal state is hidden. |
| Is the comparison fair? | Same layouts, seeds, network, settings and actual budgets; augmentation differs. |
| What if Studio stops? | Resume the last intact update with optimiser, games and RNGs. Unsaved work can repeat. |
| Does it need S3? | No. It uses the persistent Studio project folder. |
| Is the research finished? | The software exists; convergence and final multi-seed findings remain unverified. |

Practise these answers in your own words, and follow the relevant source functions for detail.

## 7. What to demonstrate

Buy a small army in the app and explain the legal perimeter choices. Show a heuristic replay,
observation layers and D4 views. Use notebook 03b to explain recovery. Label any short-run PPO
replay as a software check.

Do not promise augmentation improves performance before the matched study provides evidence.